# D2: Dashboard Data Export

---

## Overview

Generate JSON data files for website dashboards and interactive maps.

**Outputs:**
- `projects_map.json` - GeoJSON for map visualization
- `summary.json` - Dashboard summary statistics
- `timeseries.json` - Time-series data for charts

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# Add modules to path
sys.path.insert(0, str(Path.cwd().parent))

from modules.report_generator import (
    generate_status_summary,
    export_to_json,
    export_map_data,
    generate_dashboard_data
)
from modules.data_loader import load_csv

# Configuration
with open('../00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

DATA_DIR = Path(CONFIG['paths']['data_dir'])
DASHBOARD_DIR = DATA_DIR / 'dashboard'
DASHBOARD_DIR.mkdir(exist_ok=True)

print(f"Dashboard output directory: {DASHBOARD_DIR}")

## 2. Load Data

In [ ]:
# Load housing projects
housing_path = Path(CONFIG['paths']['housing_projects'])
df = load_csv(housing_path)

if df is not None:
    print(f"Loaded {len(df)} projects")
    
    # Check coordinate coverage
    has_coords = df['latitude'].notna().sum() if 'latitude' in df.columns else 0
    print(f"With coordinates: {has_coords} ({100*has_coords/len(df):.1f}%)")

## 3. Generate Map Data (GeoJSON)

In [ ]:
# Export map data
if df is not None and 'latitude' in df.columns:
    map_path = DASHBOARD_DIR / 'projects_map.json'
    export_map_data(df, map_path)
    
    # Show sample
    with open(map_path) as f:
        map_data = json.load(f)
    
    print(f"\nMap data features: {len(map_data['features'])}")
    if map_data['features']:
        print("\nSample feature:")
        print(json.dumps(map_data['features'][0], indent=2))

## 4. Generate Summary Statistics

In [ ]:
# Generate and export summary
if df is not None:
    summary = generate_status_summary(df)
    
    summary_path = DASHBOARD_DIR / 'summary.json'
    export_to_json(summary, summary_path)
    
    print("Summary Statistics:")
    print(json.dumps(summary, indent=2, default=str))

## 5. Generate Time Series Data

In [ ]:
# Time series by year
if df is not None and 'year' in df.columns:
    yearly = df.groupby('year').agg({
        'net_units': 'sum',
        'address_display': 'count'
    }).reset_index()
    yearly.columns = ['year', 'units', 'projects']
    yearly = yearly.dropna()
    yearly['year'] = yearly['year'].astype(int)
    
    # Export
    timeseries_path = DASHBOARD_DIR / 'timeseries.json'
    export_to_json(yearly.to_dict(orient='records'), timeseries_path)
    
    print("Time Series Data:")
    display(yearly)

## 6. Generate Status Breakdown

In [ ]:
# Status breakdown for pie chart
if df is not None:
    status_data = df.groupby('status').agg({
        'net_units': 'sum',
        'address_display': 'count'
    }).reset_index()
    status_data.columns = ['status', 'units', 'projects']
    status_data = status_data.sort_values('units', ascending=False)
    
    status_path = DASHBOARD_DIR / 'status_breakdown.json'
    export_to_json(status_data.to_dict(orient='records'), status_path)
    
    print("Status breakdown saved")

## 7. Generate All Dashboard Files

In [ ]:
# Generate all dashboard data at once
if df is not None:
    files = generate_dashboard_data(df, DASHBOARD_DIR)
    
    print("\nGenerated files:")
    for name, path in files.items():
        print(f"  {name}: {path}")

## 8. Verify Output Files

In [ ]:
# List all generated files
print("Dashboard Directory Contents:")
print("="*60)

for f in sorted(DASHBOARD_DIR.glob('*.json')):
    size = f.stat().st_size / 1024
    print(f"  {f.name}: {size:.1f} KB")

---

## Summary

This notebook:
- Generated GeoJSON map data
- Created summary statistics JSON
- Exported time series data
- Created status breakdown

**Next:** Run `D3_alerts_monitoring.ipynb` for project alerts.